# Three-body chaos: the Pythagorean problem

*mechanics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/three_body_chaos)

Burrau's problem: masses 3, 4 and 5 start at rest on the corners of a 3-4-5 triangle and dance chaotically until one is flung out. A twin run started $\delta = 10^{-6}$ away is integrated in the same tensor (run index $k$); the distance between the two grows exponentially, $|\Delta| \propto e^{\lambda t}$, and their fates part ways.

**Vector form**

$$\ddot{\mathbf r}_a = G\sum_{b\neq a} m_b\,\frac{\mathbf r_b - \mathbf r_a}{\lVert\mathbf r_b - \mathbf r_a\rVert^{3}},\qquad |\Delta(t)| \approx |\Delta(0)|\,e^{\lambda t}$$

**Index (tensor) form**

$$r_{kabi} = x_{kbi} - x_{kai},\qquad \ddot x_{kai} = G\,m_b\,\left(r_{kabj}r_{kabj}\right)^{-3/2}\,r_{kabi},\qquad |\Delta|^2 = \Delta x_{ai}\Delta x_{ai} + \Delta v_{ai}\Delta v_{ai}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{kai}$ | `x` | (K, 3, 2) | position of body $a$ in run $k$ ($k = 0$ reference, $k = 1$ twin), plane component $i$ |
| $m_b$ | `m` | (3,) | masses 3, 4, 5 |
| $r_{kabi}$ | `x[:, None, :, :] - x[:, :, None, :]` | (K, 3, 3, 2) | separation from $a$ to $b$, all pairs of both runs by broadcasting |
| $r_{kabj}r_{kabj}$ | `einsum('kabj,kabj->kab', r, r)` | (K, 3, 3) | repeated $j$ ⇒ summed: squared distance of every pair |
| $\ddot x_{kai}$ | `G * einsum('b,kab,kabi->kai', m, inv_d3, r)` | (K, 3, 2) | repeated $b$ ⇒ summed: total pull on body $a$; $k$ is not summed, the runs stay independent |
| $k_s$ | `dopri_step(f, y, k1, dt)` | (7, 2, K, 3, 2) | the 7 stage slopes of the adaptive Dormand–Prince step; their 5th- minus 4th-order combination is the error estimate |
| $\vert \Delta\vert $ | `phase_space_distance(xs, vs)` | (T,) | distance between twin and reference in positions and velocities |
| $\lambda$ | `lyapunov_fit(times, dist)` | () | slope of $\ln/\Delta/$ vs $t$ until the runs separate for good: the Lyapunov exponent |

Close encounters are the hard part: two bodies swing past each other at a distance of about $0.01$, where the force is $10^4$ times larger than usual. A fixed step would either blow up there or crawl everywhere else, so the solver uses the adaptive Dormand–Prince 5(4) method: each step compares a 5th- and a 4th-order answer, and dt shrinks automatically near a close pass and grows again afterwards. No softening is used, so this is exact Newtonian gravity with $G = 1$. Both runs share one step, so they are compared at equal times. Because the motion is chaotic, the tolerance matters: with the default $10^{-11}$ the reference reproduces the classic outcome (Szebehely and Peters, 1967): the lightest body ($m = 3$) is ejected around $t \approx 60$ while masses 4 and 5 leave as a tight binary in the opposite direction. The twin, moved by only $\delta$, has separated completely by $t \approx 45$ and follows a different history. The curve of $|\Delta|$ is jumpy: it spikes at every close encounter, where the twin passes a moment earlier or later at high speed, and it grows in bursts. The fitted $\lambda$ is the average trend from $t = 0$ until $|\Delta|$ stays above $0.1$ for good. Try a looser tolerance: the reference itself then changes its fate, a reminder that numerical error is a perturbation too.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/three_body_chaos/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    G, delta = tensors.G, 10.0 ** params["log10_delta"]
    m, x0, v0 = tensors.initial_state(delta, dev)  # (3,), (K=2, 3, 2), (K=2, 3, 2)

    times, xs, vs, n_steps = solve.solve(m, x0, v0, G, params["t_max"], 10.0 ** params["log10_tol"])
    dist = equation.phase_space_distance(xs, vs)  # (T,)
    fit = solve.lyapunov_fit(times, dist)
    eject = solve.ejection(xs[:, 0], vs[:, 0], m, G)  # reference run
    eject_twin = solve.ejection(xs[:, 1], vs[:, 1], m, G)

    e_err = solve.energy_error(xs, vs, m, G)[:, 0].max().item()
    com = equation.center_of_mass(xs[:, 0], m).norm(dim=-1).max().item()
    print(f"{n_steps} adaptive steps, t_max = {params['t_max']:g}, δ = {delta:.0e}")
    print(f"reference run: max relative energy error {e_err:.1e}, centre of mass moves at most {com:.1e}")
    print(f"phase-space distance: {dist[0]:.1e} -> {dist.max():.2f}; fitted Lyapunov exponent λ = {fit[0]:.3f}")
    for name, (step, body) in (("reference", eject), ("twin", eject_twin)):
        if step is None:
            print(f"{name}: no escape before t = {params['t_max']:g}")
        else:
            print(f"{name}: mass {m[body].item():g} escapes at t ≈ {times[step].item():.1f}")
    return plot.animate(times, xs, dist, m, fit, eject, out_dir / "three_body_chaos.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/three_body_chaos/equation.py
r"""Physics → tensor translation.

Newtonian gravity for three bodies in a plane, for K runs at once (run 0 = reference, run 1 = twin):

    \ddot{\mathbf r}_a = G \sum_{b \neq a} m_b (\mathbf r_b - \mathbf r_a) / \lVert \mathbf r_b - \mathbf r_a \rVert^3

In index notation, with run k, bodies a, b and plane components i, j ∈ {x, y}:

    r_{kabi} = x_{kbi} - x_{kai}                                (separation tensor, all pairs of every run)
    \ddot x_{kai} = G \, m_b \, (r_{kabj} r_{kabj})^{-3/2} \, r_{kabi}

Repeated indices are summed: j gives the squared distance of each pair, einsum('kabj,kabj->kab'),
and b adds up the pull of the other bodies on body a, einsum('b,kab,kabi->kai').
The run index k is never summed: the runs evolve side by side, independently.

Chaos is measured by the phase-space distance between two runs,

    |\Delta| = \sqrt{\Delta x_{ai}\Delta x_{ai} + \Delta v_{ai}\Delta v_{ai}},   \Delta x_{ai} = x_{1ai} - x_{0ai},

which grows like e^{\lambda t} (λ = largest Lyapunov exponent) until it is as large as the orbits themselves.
"""

import torch


def separations(x: torch.Tensor) -> torch.Tensor:
    r"""r_{kabi} = x_{kbi} - x_{kai}.   x: (K, N, 2) → (K, N, N, 2) by broadcasting."""
    return x[:, None, :, :] - x[:, :, None, :]


def pair_distances(x: torch.Tensor) -> torch.Tensor:
    r"""\sqrt{r_{kabj} r_{kabj}}.   x: (K, N, 2) → (K, N, N); the diagonal (a = b) is 0."""
    r = separations(x)
    return torch.einsum("kabj,kabj->kab", r, r).sqrt()


def acceleration(x: torch.Tensor, m: torch.Tensor, G: float) -> torch.Tensor:
    r"""\ddot x_{kai} = G m_b (r_{kabj} r_{kabj})^{-3/2} r_{kabi}.   x: (K, N, 2), m: (N,) → (K, N, 2)"""
    r = separations(x)  # (K, N, N, 2)
    dist_sq = torch.einsum("kabj,kabj->kab", r, r)  # (K, N, N)
    inv_d3 = dist_sq ** (-1.5)  # (K, N, N); the diagonal a = b is 1/0 = inf ...
    inv_d3.diagonal(dim1=1, dim2=2).zero_()  # ... and is set to 0: no self-force
    return G * torch.einsum("b,kab,kabi->kai", m, inv_d3, r)  # sum over b


def rhs(y: torch.Tensor, m: torch.Tensor, G: float) -> torch.Tensor:
    """dy/dt for y = (x, v) stacked: (2, K, N, 2) → (2, K, N, 2)."""
    x, v = y[0], y[1]
    return torch.stack([v, acceleration(x, m, G)])


def energy(x: torch.Tensor, v: torch.Tensor, m: torch.Tensor, G: float) -> torch.Tensor:
    r"""E_k = ½ m_a v_{kai} v_{kai} - ½ G Σ_{a≠b} m_a m_b / |r_{kab}|.   x, v: (..., K, N, 2) → (..., K)"""
    kinetic = 0.5 * torch.einsum("a,...ai,...ai->...", m, v, v)
    r = x[..., None, :, :] - x[..., :, None, :]  # (..., N, N, 2)
    dist = torch.einsum("...abj,...abj->...ab", r, r).sqrt()
    eye = torch.eye(len(m), dtype=torch.bool, device=x.device)
    inv_d = torch.where(eye, 0.0, 1.0 / torch.where(eye, 1.0, dist))
    potential = -0.5 * G * torch.einsum("a,b,...ab->...", m, m, inv_d)  # ½: each pair appears twice
    return kinetic + potential


def momentum(v: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    r"""P_i = m_a v_{ai} (sum over bodies a).   v: (..., N, 2) → (..., 2)"""
    return torch.einsum("a,...ai->...i", m, v)


def center_of_mass(x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    r"""X_i = m_a x_{ai} / M.   x: (..., N, 2) → (..., 2)"""
    return torch.einsum("a,...ai->...i", m, x) / m.sum()


def phase_space_distance(xs: torch.Tensor, vs: torch.Tensor) -> torch.Tensor:
    r"""|Δ| between run 1 and run 0.   xs, vs: (T, K, N, 2) → (T,)"""
    dx, dv = xs[:, 1] - xs[:, 0], vs[:, 1] - vs[:, 0]  # (T, N, 2)
    return (torch.einsum("tai,tai->t", dx, dx) + torch.einsum("tai,tai->t", dv, dv)).sqrt()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/three_body_chaos/tensors.py
"""Build the concrete tensors for Burrau's Pythagorean three-body problem.

Masses 3, 4, 5 start at rest on the corners of a 3-4-5 right triangle, each opposite the side of the
same length. Units: G = 1. float64, because the twin run differs from the reference by only δ ~ 1e-6.
"""

import torch

G = 1.0
DTYPE = torch.float64


def masses(device: torch.device | None = None) -> torch.Tensor:
    """m_a, shape (3,)."""
    return torch.tensor([3.0, 4.0, 5.0], dtype=DTYPE, device=device)


def pythagorean_positions(device: torch.device | None = None) -> torch.Tensor:
    """x_ai, shape (3, 2): the right angle sits at mass 5; the centre of mass is at the origin.

    side opposite mass 3: |x_4 - x_5| = 3,  opposite mass 4: |x_3 - x_5| = 4,  opposite mass 5: |x_3 - x_4| = 5.
    """
    return torch.tensor([[1.0, 3.0], [-2.0, -1.0], [1.0, -1.0]], dtype=DTYPE, device=device)


def initial_state(delta: float, device: torch.device | None = None) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """(m (3,), x (K=2, 3, 2), v (K=2, 3, 2)): run 0 = reference, run 1 = twin with mass 3 moved by δ along x."""
    m = masses(device)
    x = pythagorean_positions(device).expand(2, 3, 2).clone()  # (K, N, 2) two copies
    x[1, 0, 0] += delta  # the twin: one coordinate nudged by δ
    v = torch.zeros_like(x)  # everything starts at rest
    return m, x, v


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/three_body_chaos/solve.py
"""Adaptive Runge–Kutta (Dormand–Prince 5(4)): the step shrinks by itself during close encounters.

Each step computes a 5th- and a 4th-order answer from the same 7 stages; their difference estimates the
error. Steps that are too inaccurate are retried with a smaller dt, and quiet stretches get bigger steps.
"""

import math

import torch

from problems._common.progress import bar

from .equation import energy, rhs

# Butcher tableau of Dormand–Prince 5(4): stage s uses y + dt · A[s, r] k_r (sum over earlier stages r)
A = torch.tensor(
    [
        [0, 0, 0, 0, 0, 0],
        [1 / 5, 0, 0, 0, 0, 0],
        [3 / 40, 9 / 40, 0, 0, 0, 0],
        [44 / 45, -56 / 15, 32 / 9, 0, 0, 0],
        [19372 / 6561, -25360 / 2187, 64448 / 6561, -212 / 729, 0, 0],
        [9017 / 3168, -355 / 33, 46732 / 5247, 49 / 176, -5103 / 18656, 0],
        [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84],  # = the 5th-order weights
    ],
    dtype=torch.float64,
)
B5 = torch.tensor([35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0], dtype=torch.float64)
B4 = torch.tensor([5179 / 57600, 0, 7571 / 16695, 393 / 640, -92097 / 339200, 187 / 2100, 1 / 40], dtype=torch.float64)


def dopri_step(f, y: torch.Tensor, k1: torch.Tensor, dt: float, A=A, B5=B5, B4=B4):
    """One Dormand–Prince trial step. Returns (y_new, k at y_new, error estimate), all shaped like y."""
    k = [k1]  # the stage slopes k_s, each shaped like y
    for s in range(1, 7):
        k.append(f(y + dt * torch.tensordot(A[s, :s], torch.stack(k), dims=1)))  # sum over earlier stages r
    k = torch.stack(k)  # (7, ...)
    y_new = y + dt * torch.tensordot(B5, k, dims=1)
    err = dt * torch.tensordot(B5 - B4, k, dims=1)  # 5th − 4th order answer
    return y_new, k[6], err  # k[6] was evaluated at y_new: reused as the next k1 ("first same as last")


def solve(
    m: torch.Tensor, x0: torch.Tensor, v0: torch.Tensor, G: float, t_max: float, tol: float, dt_out: float = 0.05
):
    """Return times (T,), positions (T, K, N, 2), velocities (T, K, N, 2) every dt_out, and the accepted step count.

    All runs share one step (the error is the largest over every run), so they stay on the same clock.
    Steps are trimmed so that they land exactly on each output time.
    """
    n_out = int(round(t_max / dt_out))
    y = torch.stack([x0, v0])  # (2, K, N, 2)
    f = lambda y: rhs(y, m, G)  # noqa: E731
    tableau = A.to(y.device), B5.to(y.device), B4.to(y.device)
    k1 = f(y)
    ys, t, dt, n_steps = [y], 0.0, 1e-3, 0
    with bar(n_out, "output times") as progress:
        for j in range(n_out):
            t_next = (j + 1) * dt_out
            while t < t_next - 1e-12:
                h = min(dt, t_next - t)
                y_new, k_new, err = dopri_step(f, y, k1, h, *tableau)
                ratio = (err.abs().max() / (tol * (1 + y.abs().max()))).item()  # ≤ 1 means accurate enough
                if ratio <= 1:  # accept
                    y, k1, t = y_new, k_new, t + h
                    n_steps += 1
                dt_new = h * min(5.0, max(0.2, 0.9 * ratio ** (-1 / 5))) if ratio > 0 else 5 * h  # error ∝ dt^5
                trimmed = h < dt and ratio <= 1  # a short step to hit t_next says nothing about the right dt
                dt = max(dt, dt_new) if trimmed else dt_new
            ys.append(y)
            progress.update(1)
    traj = torch.stack(ys)  # (T, 2, K, N, 2)
    times = torch.arange(n_out + 1, dtype=x0.dtype, device=x0.device) * dt_out
    return times, traj[:, 0], traj[:, 1], n_steps


def lyapunov_fit(times: torch.Tensor, dist: torch.Tensor, high: float = 0.1):
    r"""Fit ln|Δ| = c + λ t by least squares, from t = 0 until |Δ| exceeds `high` for good.

    |Δ| spikes at every close encounter (the twin passes a moment earlier or later), so the window ends
    only once the runs have separated for good: from then on |Δ| never drops below `high` again.
    Returns (λ, t_start, t_end, c); λ is nan if the runs never separate (e.g. δ = 0).
    """
    stays_above = dist.flip(0).cummin(0).values.flip(0) > high  # (T,) True from the point of no return on
    if dist[0] <= 0 or not stays_above.any():
        return math.nan, math.nan, math.nan, math.nan
    i1 = stays_above.nonzero()[0].item()
    t, y = times[: i1 + 1], dist[: i1 + 1].log()
    X = torch.stack([torch.ones_like(t), t], dim=1)  # (n, 2) least squares for [c, λ]
    c, lam = torch.linalg.lstsq(X, y[:, None]).solution[:, 0].tolist()
    return lam, t[0].item(), t[-1].item(), c


def ejection(xs: torch.Tensor, vs: torch.Tensor, m: torch.Tensor, G: float, r_min: float = 5.0):
    """First output step at which one body has escaped from the other two, as (step, body) or (None, None).

    Escape: the body is farther than r_min from the centre of mass of the other two, and its two-body energy
    relative to that pair (treated as one mass) is positive, so it never comes back.  xs, vs: (T, N, 2)
    """
    for a in range(len(m)):
        others = [b for b in range(len(m)) if b != a]
        M = m[others].sum()
        x_pair = torch.einsum("b,tbi->ti", m[others], xs[:, others]) / M  # (T, 2) centre of mass of the pair
        v_pair = torch.einsum("b,tbi->ti", m[others], vs[:, others]) / M
        r = (xs[:, a] - x_pair).norm(dim=-1)  # (T,)
        u = (vs[:, a] - v_pair).norm(dim=-1)
        e_rel = 0.5 * u**2 - G * (m[a] + M) / r  # energy per reduced mass
        gone = ((r > r_min) & (e_rel > 0)).nonzero()
        if len(gone):
            return gone[0].item(), a
    return None, None


def energy_error(xs: torch.Tensor, vs: torch.Tensor, m: torch.Tensor, G: float) -> torch.Tensor:
    """|E(t) − E(0)| / |E(0)| for every run.   (T, K, N, 2) → (T, K)"""
    e = energy(xs, vs, m, G)
    return (e - e[0]).abs() / e[0].abs()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/three_body_chaos/plot.py
"""Left: the three bodies of the reference run (solid) and the twin (dashed). Right: log10 of their distance."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

COLORS = ["#2563eb", "#dc2626", "#16a34a"]  # masses 3, 4, 5


def animate(
    times: torch.Tensor,
    xs: torch.Tensor,
    dist: torch.Tensor,
    m: torch.Tensor,
    fit: tuple,
    eject: tuple,
    path: Path,
    trail: float = 8.0,
) -> Path:
    """xs: (T, K, N, 2); dist: (T,) phase-space distance; fit = (λ, t0, t1, c); eject = (step, body) of run 0.

    trail = how many time units of path each body drags behind it.
    """
    t = times.cpu().numpy()  # (T,)
    pos = xs.cpu().numpy()  # (T, K, N, 2)
    log_d = dist.clamp_min(1e-300).log10().cpu().numpy()  # (T,)
    mass = m.cpu().numpy()
    tail = max(2, int(trail / (t[1] - t[0])))  # trail length in stored steps
    frames = frame_indices(len(t), 90)
    reach = np.maximum.accumulate(abs(pos).max(axis=(1, 2, 3)))  # (T,) farthest any body has been so far
    reach = 1.08 * np.maximum(reach, 4.0)  # zoom out only when a body flies away

    fig, (ax, ax_d) = plt.subplots(1, 2, figsize=(7.6, 4), gridspec_kw={"width_ratios": [1, 1.15]})

    # --- left: orbits ---
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])  # the view zooms out during the escape; ticks would be redrawn every frame
    ax.set_yticks([])
    ax.set_title("solid: reference   dashed: twin (+δ)", fontsize=9)
    ref_lines = [ax.plot([], [], lw=1.1, color=c)[0] for c in COLORS]
    twin_lines = [ax.plot([], [], lw=1.0, ls="--", color=c, alpha=0.45)[0] for c in COLORS]
    sizes = 4 + 2.5 * mass / mass.max() * 2
    ref_dots = [ax.plot([], [], "o", ms=s, color=c, label=f"m = {mi:g}")[0] for s, c, mi in zip(sizes, COLORS, mass)]
    twin_dots = [ax.plot([], [], "o", ms=s, mfc="none", color=c, alpha=0.6)[0] for s, c in zip(sizes, COLORS)]
    ax.legend(loc="lower left", fontsize=7, framealpha=0.8)
    clock = ax.text(0.03, 0.97, "", transform=ax.transAxes, va="top", fontsize=9)
    escape_note = ax.text(0.97, 0.03, "", transform=ax.transAxes, va="bottom", ha="right", fontsize=8, color="#b45309")

    # --- right: separation of the two runs in powers of ten (straight rise = exponential growth) ---
    ax_d.set_xlim(t[0], t[-1])
    ax_d.set_ylim(log_d.min() - 0.5, max(log_d.max(), 0.0) + 0.8)
    ax_d.set_xlabel("t")
    ax_d.set_ylabel("log₁₀ |Δ|   (phase-space distance)")
    trace = ax_d.plot([], [], lw=1.1, color="#111827")[0]
    lam, t0, t1, c = fit
    if lam == lam:  # not nan
        ax_d.plot([t0, t1], [(c + lam * t0) / 2.302585, (c + lam * t1) / 2.302585], "--", color="#f59e0b", lw=2)
        ax_d.text(0.03, 0.97, f"fit: λ ≈ {lam:.2f}", transform=ax_d.transAxes, va="top", color="#b45309")
    step_ej, body_ej = eject
    if step_ej is not None:
        pair = " + ".join(f"{mi:g}" for mi in mass if mi != mass[body_ej])  # the two that stay together
        ax_d.axvline(t[step_ej], color="#9ca3af", ls=":", lw=1.5)
        ax_d.text(
            t[step_ej],
            ax_d.get_ylim()[0] + 0.3,
            f"m = {mass[body_ej]:g} escapes ",
            ha="right",
            fontsize=8,
            color="#374151",
        )

    def update(step: int):
        start = max(0, step - tail)
        for a in range(len(mass)):
            ref_lines[a].set_data(pos[start : step + 1, 0, a, 0], pos[start : step + 1, 0, a, 1])
            twin_lines[a].set_data(pos[start : step + 1, 1, a, 0], pos[start : step + 1, 1, a, 1])
            ref_dots[a].set_data([pos[step, 0, a, 0]], [pos[step, 0, a, 1]])
            twin_dots[a].set_data([pos[step, 1, a, 0]], [pos[step, 1, a, 1]])
        ax.set_xlim(-reach[step], reach[step])
        ax.set_ylim(-reach[step], reach[step])
        clock.set_text(f"t = {t[step]:.1f}")
        if step_ej is not None and step >= step_ej:
            escape_note.set_text(f"m = {mass[body_ej]:g} escapes,\nbinary {pair} left")
        trace.set_data(t[: step + 1], log_d[: step + 1])

    fig.tight_layout()
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    't_max': 63.0,
    'log10_delta': -6,
    'log10_tol': -11
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.three_body_chaos")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.three_body_chaos.main")
problem_dir = Path("problems/mechanics/three_body_chaos")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
